Load one season and find Texans games

In [1]:
import polars as pl

# Show more rows/columns when displaying a DataFrame (defaults are small)
pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(25)

# Load just ONE season. We're exploring, so we don't need all 770k rows.
# The path starts with ../ because the notebook runs from the notebooks/ folder,
# so we step up one level to the project root first.
pbp = pl.read_parquet("../data/raw/pbp_2025.parquet")

# Find every 2025 game the Texans played (home OR away).
# Every play in a game repeats the same game info, so .unique()
# collapses ~150 rows per game down to one row per game.
hou_games = (
    pbp.filter((pl.col("home_team") == "HOU") | (pl.col("away_team") == "HOU"))
    .select(
        "game_id", "week", "home_team", "away_team",
        "home_score", "away_score",   # final scores
        "spread_line",                # pregame point spread
        "result",                     # final margin
    )
    .unique()
    .sort("week")
)
hou_games

game_id,week,home_team,away_team,home_score,away_score,spread_line,result
str,i32,str,str,i32,i32,f64,i32
"""2025_01_HOU_LA""",1,"""LA""","""HOU""",14,9,3.5,5
"""2025_02_TB_HOU""",2,"""HOU""","""TB""",19,20,2.5,-1
"""2025_03_HOU_JAX""",3,"""JAX""","""HOU""",17,10,1.5,7
"""2025_04_TEN_HOU""",4,"""HOU""","""TEN""",26,0,7.5,26
"""2025_05_HOU_BAL""",5,"""BAL""","""HOU""",10,44,-2.5,-34
"""2025_07_HOU_SEA""",7,"""SEA""","""HOU""",27,19,3.0,8
"""2025_08_SF_HOU""",8,"""HOU""","""SF""",26,15,2.5,11
"""2025_09_DEN_HOU""",9,"""HOU""","""DEN""",15,18,1.5,-3
"""2025_10_JAX_HOU""",10,"""HOU""","""JAX""",36,29,-1.5,7


Pull out that game's key columns

In [5]:
GAME_ID = "2025_02_TB_HOU"

# Keep only this game, and only the columns we care about for the model
game = (
    pbp.filter(pl.col("game_id") == GAME_ID)
    .select(
        "qtr",                          # quarter (5 = overtime)
        "game_seconds_remaining",       # time left in the whole game
        "posteam", "defteam",           # who has the ball / who's defending
        "posteam_score", "defteam_score",
        "score_differential",           # score from someone's point of view (which?)
        "down", "ydstogo",              # down and yards to go for a first down
        "yardline_100",                 # field position (measured how?)
        "posteam_timeouts_remaining",
        "defteam_timeouts_remaining",
        "play_type",                    # pass, run, punt, kickoff, no_play...
        "wp",                           # nflfastR's own win probability (our future rival)
        "desc",                         # plain-English description of the play
    )
)

game.head(30)   # the opening of the game

qtr,game_seconds_remaining,posteam,defteam,posteam_score,defteam_score,score_differential,down,ydstogo,yardline_100,posteam_timeouts_remaining,defteam_timeouts_remaining,play_type,wp,desc
f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str
1.0,3600.0,null,null,null,null,null,null,0.0,null,null,null,null,0.546262,"""GAME"""
1.0,3600.0,"""HOU""","""TB""",0.0,0.0,0.0,null,0.0,35.0,3.0,3.0,"""kickoff""",0.546262,"""4-C.McLaughlin kicks 63 yards …"
1.0,3593.0,"""HOU""","""TB""",0.0,0.0,0.0,1.0,10.0,64.0,3.0,3.0,"""pass""",0.572573,"""(14:53) 7-C.Stroud pass incomp…"
1.0,3589.0,"""HOU""","""TB""",0.0,0.0,0.0,2.0,10.0,64.0,3.0,3.0,"""pass""",0.554537,"""(14:49) (Shotgun) 7-C.Stroud p…"
1.0,3545.0,"""HOU""","""TB""",0.0,0.0,0.0,3.0,1.0,55.0,3.0,3.0,"""run""",0.563459,"""(14:05) 21-N.Chubb up the midd…"
1.0,3500.0,"""HOU""","""TB""",0.0,0.0,0.0,1.0,10.0,54.0,3.0,3.0,"""run""",0.579976,"""(13:20) 21-N.Chubb up the midd…"
1.0,3465.0,"""HOU""","""TB""",0.0,0.0,0.0,2.0,12.0,56.0,3.0,3.0,"""pass""",0.553961,"""(12:45) (Shotgun) 7-C.Stroud p…"
1.0,3418.0,"""HOU""","""TB""",0.0,0.0,0.0,1.0,10.0,29.0,3.0,3.0,"""pass""",0.64672,"""(11:58) (Shotgun) 7-C.Stroud p…"
1.0,3413.0,"""HOU""","""TB""",6.0,0.0,6.0,null,0.0,15.0,3.0,3.0,"""extra_point""",0.697489,"""15-K.Fairbairn extra point is …"


In [6]:
game.tail(25)   # the finish

qtr,game_seconds_remaining,posteam,defteam,posteam_score,defteam_score,score_differential,down,ydstogo,yardline_100,posteam_timeouts_remaining,defteam_timeouts_remaining,play_type,wp,desc
f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str
4.0,283.0,"""TB""","""HOU""",14.0,13.0,1.0,2.0,10.0,65.0,3.0,3.0,"""pass""",0.639888,"""(4:43) (Shotgun) 6-B.Mayfield …"
4.0,240.0,"""TB""","""HOU""",14.0,13.0,1.0,3.0,8.0,63.0,3.0,3.0,"""pass""",0.61477,"""(4:00) (Shotgun) 6-B.Mayfield …"
4.0,198.0,"""TB""","""HOU""",14.0,13.0,1.0,4.0,16.0,71.0,3.0,3.0,"""punt""",0.511089,"""(3:18) 9-R.Dixon punts 50 yard…"
4.0,180.0,"""HOU""","""TB""",13.0,14.0,-1.0,1.0,10.0,26.0,3.0,3.0,"""run""",0.676771,"""(3:00) 21-N.Chubb left tackle …"
4.0,136.0,"""HOU""","""TB""",13.0,14.0,-1.0,2.0,9.0,25.0,3.0,3.0,"""run""",0.668162,"""(2:16) 21-N.Chubb right guard …"
4.0,130.0,"""HOU""","""TB""",19.0,14.0,5.0,null,0.0,2.0,3.0,3.0,"""pass""",0.834925,"""TWO-POINT CONVERSION ATTEMPT. …"
4.0,130.0,"""TB""","""HOU""",14.0,19.0,-5.0,null,0.0,35.0,3.0,3.0,"""kickoff""",0.204471,"""15-K.Fairbairn kicks 60 yards …"
4.0,124.0,"""TB""","""HOU""",14.0,19.0,-5.0,1.0,10.0,80.0,3.0,3.0,"""pass""",0.195345,"""(2:04) (Shotgun) 6-B.Mayfield …"
4.0,116.0,"""TB""","""HOU""",14.0,19.0,-5.0,1.0,10.0,68.0,3.0,3.0,"""pass""",0.223038,"""(1:56) (Shotgun) 6-B.Mayfield …"
